## Patient Speaker Annotation Pipeline

Annotates the transcript word/segment DataFrame with `patient_speaking` flags derived from LR-ASD video diarization scores, then generates subtitled validation clips.

### 1. Imports

In [1]:
from glob import glob
import json
import os
import re
import subprocess
import tempfile
from pathlib import Path

import cv2
import dill as pickle
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

### 2. Configuration

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
PATIENT        = "YFG"
VAD_NEW_DIR    = "/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new"
PATIENT_DIR    = f"{VAD_NEW_DIR}/{PATIENT}"
TRANSCRIPT_CSV = f"{PATIENT_DIR}/{PATIENT}_transcripts.csv"
OUTPUT_CSV     = f"{PATIENT_DIR}/{PATIENT}_transcripts_annotated.csv"
SAMPLES_DIR    = f"{PATIENT_DIR}/patient_speech_samples"

# ── LR-ASD parameters (tunable) ───────────────────────────────────────────────
CAMERA_SERIAL    = "18486638"  # camera serial used for video sync
FPS              = 25.0        # LR-ASD output FPS
SMOOTH_MS        = 400.0       # smoothing window in ms
LRASD_THRESHOLD  = 0.5         # min smoothed LR-ASD score to count a frame as speaking
PROP_THRESHOLD   = 0.5         # min proportion of segment frames where patient speaks
MIN_SEG_DURATION = 2.0         # minimum segment duration (seconds) to consider

os.makedirs(SAMPLES_DIR, exist_ok=True)

### 3. LR-ASD Helper Functions

In [3]:
def moving_average_nan(x, window):
    x = np.asarray(x, dtype=float)
    valid = np.isfinite(x).astype(float)
    x_filled = np.nan_to_num(x, nan=0.0)
    kernel = np.ones(window, dtype=float)
    num = np.convolve(x_filled, kernel, mode='same')
    den = np.convolve(valid, kernel, mode='same')
    out = np.full_like(x, np.nan, dtype=float)
    keep = den > 0
    out[keep] = num[keep] / den[keep]
    return out


def merge_lrasd_tracks(tracks, scores, frame_width, frame_height,
                       fps=25.0, smooth_ms=400.0, threshold=0.5):
    """
    Build a per-frame DataFrame from LR-ASD tracks/scores for one interval.

    Patient heuristic: active face closest to frame centre.

    Returns columns: frame_idx, time_sec, patient_track,
                     patient_raw_score, patient_smooth_score, speaker
    """
    if len(tracks) != len(scores):
        raise ValueError(f'tracks/scores length mismatch: {len(tracks)} vs {len(scores)}')

    max_frame = max(
        (int(np.asarray(tr['track']['frame']).max())
         for tr in tracks if len(tr['track']['frame']) > 0),
        default=-1,
    )
    if max_frame < 0:
        raise ValueError('No frames found in tracks')

    n_frames = max_frame + 1
    n_tracks = len(tracks)
    raw_mat  = np.full((n_tracks, n_frames), np.nan, dtype=float)
    x_mat    = np.full((n_tracks, n_frames), np.nan, dtype=float)
    y_mat    = np.full((n_tracks, n_frames), np.nan, dtype=float)

    for tidx, (tr, sc) in enumerate(zip(tracks, scores)):
        frames = np.asarray(tr['track']['frame'], dtype=int)
        sc     = np.asarray(sc, dtype=float)
        xs     = np.asarray(tr['proc_track']['x'], dtype=float)
        ys     = np.asarray(tr['proc_track']['y'], dtype=float)
        n      = min(len(frames), len(sc), len(xs), len(ys))
        raw_mat[tidx, frames[:n]] = sc[:n]
        x_mat[tidx, frames[:n]]   = xs[:n]
        y_mat[tidx, frames[:n]]   = ys[:n]

    window = max(1, int(round(smooth_ms / 1000.0 * fps)))
    if window % 2 == 0:
        window += 1
    smooth_mat = np.vstack([moving_average_nan(raw_mat[i], window) for i in range(n_tracks)])

    cx, cy   = frame_width / 2.0, frame_height / 2.0
    dist_mat = np.sqrt((x_mat - cx) ** 2 + (y_mat - cy) ** 2)
    dist_mat[~np.isfinite(raw_mat)] = np.nan

    patient_track  = np.full(n_frames, -1, dtype=int)
    patient_raw    = np.full(n_frames, np.nan, dtype=float)
    patient_smooth = np.full(n_frames, np.nan, dtype=float)
    speaker        = np.zeros(n_frames, dtype=bool)

    for f in range(n_frames):
        active = np.where(np.isfinite(dist_mat[:, f]))[0]
        if len(active) == 0:
            continue
        best = active[np.argmin(dist_mat[active, f])]
        patient_track[f]  = int(best)
        patient_raw[f]    = raw_mat[best, f]
        patient_smooth[f] = smooth_mat[best, f]
        if np.isfinite(patient_smooth[f]) and patient_smooth[f] >= threshold:
            speaker[f] = True

    return pd.DataFrame({
        'frame_idx':            np.arange(n_frames, dtype=int),
        'time_sec':             np.arange(n_frames, dtype=float) / fps,
        'patient_track':        patient_track,
        'patient_raw_score':    patient_raw,
        'patient_smooth_score': patient_smooth,
        'speaker':              speaker,
    })

### 4. Discover LR-ASD Outputs per Interval

In [4]:
LRASD_BASE    = f'{PATIENT_DIR}/vad_data'
SYNCED_SUB    = f'neural/{CAMERA_SERIAL}/synced_video/neural_{CAMERA_SERIAL}'


def lrasd_paths(interval_id):
    base = f'{LRASD_BASE}/{interval_id}/video/{interval_id}/{SYNCED_SUB}'
    return {
        'video':   f'{base}/pyavi/video_out.avi',
        'scores':  f'{base}/pywork/scores.pckl',
        'tracks':  f'{base}/pywork/tracks.pckl',
        'frames':  f'{base}/pyframes',
        'quality': f'{os.path.dirname(base)}/sync_quality.json',
    }


def is_good_quality(quality_json_path):
    try:
        with open(quality_json_path) as f:
            return json.load(f).get('status', '').lower() == 'good'
    except Exception:
        return False


score_files = glob(
    f'{LRASD_BASE}/*/video/*/neural/{CAMERA_SERIAL}/synced_video/'
    f'neural_{CAMERA_SERIAL}/pywork/scores.pckl'
)

valid_intervals = {}
skipped = []
for sf_path in score_files:
    m = re.search(r'vad_data/([^/]+)/video', sf_path)
    if not m:
        continue
    iid = m.group(1)
    paths = lrasd_paths(iid)
    if not os.path.exists(paths['tracks']):
        skipped.append((iid, 'missing tracks.pckl'))
        continue
    if not os.path.exists(paths['video']):
        skipped.append((iid, 'missing video_out.avi'))
        continue
    if not is_good_quality(paths['quality']):
        skipped.append((iid, 'bad sync quality'))
        continue
    valid_intervals[iid] = paths

print(f'Found {len(valid_intervals)} intervals with complete, good-quality LR-ASD outputs')
print(f'Skipped {len(skipped)} intervals')
if skipped:
    reason_counts = {}
    for _, r in skipped:
        reason_counts[r] = reason_counts.get(r, 0) + 1
    print('  Skip reasons:', reason_counts)

Found 269 intervals with complete, good-quality LR-ASD outputs
Skipped 0 intervals


### 5. Get Video Dimensions

In [5]:
_frame_W, _frame_H = None, None
for iid, paths in valid_intervals.items():
    frame_files = sorted(glob(f"{paths['frames']}/*.jpg"))
    if not frame_files:
        continue
    img = cv2.imread(frame_files[0])
    if img is None:
        continue
    _frame_H, _frame_W = img.shape[:2]
    print(f'Frame dimensions: {_frame_W} x {_frame_H} (from {iid})')
    break

assert _frame_W is not None, 'Could not read any frame to determine video dimensions'

Frame dimensions: 1920 x 1200 (from 20240923-200755_0050)


### 6. Build Per-Interval Patient-Speaking DataFrames

In [6]:
lrasd_dfs = {}

for iid, paths in tqdm(valid_intervals.items(), desc='Loading LR-ASD outputs'):
    try:
        with open(paths['scores'], 'rb') as f:
            scores = pickle.load(f)
        with open(paths['tracks'], 'rb') as f:
            tracks = pickle.load(f)
        df = merge_lrasd_tracks(
            tracks=tracks, scores=scores,
            frame_width=_frame_W, frame_height=_frame_H,
            fps=FPS, smooth_ms=SMOOTH_MS, threshold=LRASD_THRESHOLD,
        )
        lrasd_dfs[iid] = df
    except Exception:
        pass  # intervals with no detectable faces produce no DF

print(f'Built speaking DFs for {len(lrasd_dfs)} / {len(valid_intervals)} intervals')

Loading LR-ASD outputs:   0%|          | 0/269 [00:00<?, ?it/s]

Built speaking DFs for 132 / 269 intervals


### 7. Load Transcript DataFrame

In [7]:
transcript_df = pd.read_csv(TRANSCRIPT_CSV)
print(f'Loaded {len(transcript_df):,} word rows across {transcript_df["interval_id"].nunique()} intervals')
transcript_df.head(3)

Loaded 56,221 word rows across 452 intervals


,patient,interval_id,toc,word,word_start_s,word_end_s,word_score,segment_idx,segment_text,segment_start_s,...,interval_utc_start,interval_utc_end,interval_dur_s,br_ts,br_te,utc_word_start,utc_word_end,br_word_start,br_word_end,context
0,YFG,20240923-200755_0000,20240923-200755,You,0.092,0.194,0.316,0,"You know, I just started.",0.092,...,2024-09-24 01:08:15.503,2024-09-24 01:08:17.403,1.9,29572790,29629790,2024-09-24 01:08:15.595000000,2024-09-24 01:08:15.697000000,29575550,29578610,NaN
1,YFG,20240923-200755_0000,20240923-200755,"know,",0.234,0.376,0.351,0,"You know, I just started.",0.092,...,2024-09-24 01:08:15.503,2024-09-24 01:08:17.403,1.9,29572790,29629790,2024-09-24 01:08:15.737000000,2024-09-24 01:08:15.879000000,29579810,29584070,You
2,YFG,20240923-200755_0000,20240923-200755,I,0.397,0.417,0.001,0,"You know, I just started.",0.092,...,2024-09-24 01:08:15.503,2024-09-24 01:08:17.403,1.9,29572790,29629790,2024-09-24 01:08:15.900000000,2024-09-24 01:08:15.920000000,29584700,29585300,"You know,"


### 8. Annotate Segments with `patient_speaking`

In [8]:
transcript_df['patient_speaking'] = False

groups = transcript_df.groupby(['interval_id', 'segment_idx'])
annotated, skipped_no_lrasd, skipped_short = 0, 0, 0

for (iid, seg_idx), group in tqdm(groups, desc='Annotating segments'):
    if iid not in lrasd_dfs:
        skipped_no_lrasd += 1
        continue

    row0      = group.iloc[0]
    seg_start = row0['segment_start_s']
    seg_end   = row0['segment_end_s']
    seg_dur   = seg_end - seg_start

    if seg_dur < MIN_SEG_DURATION:
        skipped_short += 1
        continue

    ldf  = lrasd_dfs[iid]
    mask = (ldf['time_sec'] >= seg_start) & (ldf['time_sec'] <= seg_end)
    seg_frames = ldf[mask]

    if len(seg_frames) == 0:
        continue

    prop_speaking = seg_frames['speaker'].sum() / len(seg_frames)
    if prop_speaking >= PROP_THRESHOLD:
        transcript_df.loc[group.index, 'patient_speaking'] = True
        annotated += 1

n_patient_words = transcript_df['patient_speaking'].sum()
print(f'Annotated {annotated:,} segments as patient speech')
print(f'  -> {n_patient_words:,} word rows marked patient_speaking=True')
print(f'  Skipped (no LR-ASD data): {skipped_no_lrasd:,} segments')
print(f'  Skipped (too short <{MIN_SEG_DURATION}s): {skipped_short:,} segments')

Annotating segments:   0%|          | 0/8133 [00:00<?, ?it/s]

Annotated 4 segments as patient speech
  -> 38 word rows marked patient_speaking=True
  Skipped (no LR-ASD data): 6,317 segments
  Skipped (too short <2.0s): 1,087 segments


### 9. Save Annotated Transcript CSV

In [9]:
transcript_df.to_csv(OUTPUT_CSV)
print(f'Saved annotated transcript to: {OUTPUT_CSV}')

Saved annotated transcript to: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFG/YFG_transcripts_annotated.csv


---
## Validation: Patient Speech Clips with Burned Subtitles

Sample random patient-speaking segments, extract video clips from the LR-ASD output (which already has face-detection boxes overlaid), and burn word-level subtitles to validate transcript alignment.

### 10. Sampling Configuration

In [10]:
N_SAMPLES    = 20    # number of random clips to generate
RANDOM_SEED  = 46    # change for a different random sample
CONTEXT_PAD  = 1.0   # seconds of context padding on each side of the segment

np.random.seed(RANDOM_SEED)

### 11. Subtitle & Clip Generation Utilities

In [11]:
def secs_to_srt_time(s):
    h = int(s // 3600)
    m = int((s % 3600) // 60)
    sec = s % 60
    return f'{h:02d}:{m:02d}:{int(sec):02d},{int((sec % 1) * 1000):03d}'


def build_srt(words_df, clip_start_s):
    """
    Build SRT subtitle string from word rows.
    word_start_s / word_end_s are interval-relative; clip_start_s converts to clip-relative.
    """
    lines = []
    for i, (_, row) in enumerate(words_df.iterrows(), start=1):
        t_start = max(0.0, row['word_start_s'] - clip_start_s)
        t_end   = max(t_start + 0.05, row['word_end_s'] - clip_start_s)
        lines.append(
            f'{i}\n'
            f'{secs_to_srt_time(t_start)} --> {secs_to_srt_time(t_end)}\n'
            f"{row['word']}\n"
        )
    return '\n'.join(lines)


def render_subtitled_clip(video_path, clip_start_s, clip_end_s, srt_content, output_path):
    """
    Cut [clip_start_s, clip_end_s] from video_path, burn SRT subtitles, write to output_path.

    Two-pass strategy:
      1. Stream-copy cut (fast) -> temp AVI
      2. Subtitle burn (re-encode video only) -> final MP4

    Word timestamps in srt_content must be relative to clip_start_s (i.e. clip-relative).
    """
    with tempfile.NamedTemporaryFile(suffix='.avi', delete=False) as tv, \
         tempfile.NamedTemporaryFile(suffix='.srt', delete=False, mode='w') as ts:
        tmp_vid = tv.name
        tmp_srt = ts.name
        ts.write(srt_content)

    try:
        # Pass 1: fast seek + stream-copy cut
        subprocess.run([
            'ffmpeg', '-y',
            '-ss', str(clip_start_s),
            '-i', video_path,
            '-t', str(clip_end_s - clip_start_s),
            '-c', 'copy',
            tmp_vid,
        ], capture_output=True, check=True)

        # Pass 2: burn subtitles (re-encode video stream only)
        # Escape colons in path for ffmpeg filter syntax
        srt_escaped = tmp_srt.replace(':', '\\:')
        subprocess.run([
            'ffmpeg', '-y',
            '-i', tmp_vid,
            '-vf', (
                f'subtitles={srt_escaped}:'
                "force_style='FontSize=22,PrimaryColour=&H00FFFFFF,"
                "BackColour=&H80000000,Bold=1,Outline=1'"
            ),
            '-c:v', 'libx264', '-crf', '23',
            '-c:a', 'aac',
            output_path,
        ], capture_output=True, check=True)
    finally:
        for p in (tmp_vid, tmp_srt):
            try:
                os.unlink(p)
            except OSError:
                pass

### 12. Sample & Render Subtitled Clips

In [12]:
# Get segments that have LR-ASD coverage
patient_seg_df = (
    transcript_df[transcript_df['patient_speaking']]
    .groupby(['interval_id', 'segment_idx'])
    .first()
    .reset_index()
)
patient_seg_df = patient_seg_df[patient_seg_df['interval_id'].isin(valid_intervals)]

n_available = len(patient_seg_df)
n_draw = min(N_SAMPLES, n_available)
sampled = patient_seg_df.sample(n=n_draw, random_state=RANDOM_SEED).reset_index(drop=True)
print(f'Sampling {n_draw} clips from {n_available} available patient-speaking segments')

failed = []
for _, seg_row in tqdm(sampled.iterrows(), total=len(sampled), desc='Rendering clips'):
    iid       = seg_row['interval_id']
    seg_idx   = int(seg_row['segment_idx'])
    seg_start = seg_row['segment_start_s']
    seg_end   = seg_row['segment_end_s']

    clip_start = max(0.0, seg_start - CONTEXT_PAD)
    clip_end   = seg_end + CONTEXT_PAD

    # All words in this segment, sorted by time
    seg_words = transcript_df[
        (transcript_df['interval_id'] == iid) &
        (transcript_df['segment_idx'] == seg_idx)
    ].sort_values('word_start_s')

    srt_content = build_srt(seg_words, clip_start)
    video_path  = valid_intervals[iid]['video']
    out_path    = f'{SAMPLES_DIR}/{iid}_seg{seg_idx:04d}_seed{RANDOM_SEED}.mp4'

    try:
        render_subtitled_clip(video_path, clip_start, clip_end, srt_content, out_path)
    except subprocess.CalledProcessError as e:
        failed.append((iid, seg_idx, e.stderr.decode()[-300:]))

print(f'Saved {n_draw - len(failed)} clips to: {SAMPLES_DIR}/')
if failed:
    print(f'Failed ({len(failed)}):')
    for iid, seg_idx, err in failed:
        print(f'  {iid} seg{seg_idx}: ...{err}')

Sampling 4 clips from 4 available patient-speaking segments


Rendering clips:   0%|          | 0/4 [00:00<?, ?it/s]

Saved 4 clips to: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFG/patient_speech_samples/


### 13. (Optional) Concatenate Sample Clips into One Review Video

In [13]:
# Re-run cells 10-12 with a different RANDOM_SEED for a new random sample.
# This cell stitches all clips in SAMPLES_DIR that match the current seed into one video.

seed_clips = sorted(glob(f'{SAMPLES_DIR}/*_seed{RANDOM_SEED}.mp4'))
if not seed_clips:
    print('No clips found for current seed. Run cells 10-12 first.')
else:
    concat_list = f'{SAMPLES_DIR}/_concat_seed{RANDOM_SEED}.txt'
    with open(concat_list, 'w') as f:
        for p in seed_clips:
            f.write(f"file '{p}'\n")

    review_path = f'{SAMPLES_DIR}/review_seed{RANDOM_SEED}.mp4'
    subprocess.run([
        'ffmpeg', '-y',
        '-f', 'concat', '-safe', '0',
        '-i', concat_list,
        '-c', 'copy',
        review_path,
    ], capture_output=True, check=True)
    print(f'Review video ({len(seed_clips)} clips) saved to: {review_path}')

Review video (10 clips) saved to: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFG/patient_speech_samples/review_seed46.mp4
